In [1]:
import sys
from pathlib import Path

# Deixa a pasta notebooks/ importável, onde ficam os módulos de apoio.
sys.path.insert(0, str(Path.cwd() if (Path.cwd() / "apoio").exists() else Path.cwd() / "notebooks"))

from apoio import identidade as ui
from apoio.banco import Banco, RAIZ, mensagem_erro

ui.capa(
    "ETL PNAD Covid-19 em",
    "SQL Server",
    "Este notebook percorre o projeto na ordem em que ele foi construído: ambiente, banco, log, "
    "functions, staging, Data Warehouse, testes e análises. Cada parte explica a decisão e "
    "executa o SQL do repositório de verdade, mostrando o resultado logo abaixo.",
    ["<b>15</b> scripts SQL", "<b>20</b> functions", "<b>9</b> procedures", "<b>28</b> verificações", "dados <b>IBGE 2020</b>"],
)

## Antes de começar

O notebook conversa com o SQL Server do `docker-compose.yml`. Na raiz do projeto:

```bash
cp .env.example .env          # Windows: Copy-Item .env.example .env
docker compose up -d
./scripts/criar_venv.sh       # Windows: .\scripts\criar_venv.ps1
```

Depois abra este arquivo no Jupyter Lab ou no VS Code e escolha o kernel **PNAD Covid (venv)**.

Rode as células de cima para baixo. Tudo é idempotente: pode executar de novo quantas vezes quiser. A carga dos dados reais do IBGE (parte 13) fica desligada até você mudar uma variável, porque baixa cerca de 80 MB.

In [2]:
ui.secao(1, "Conexão com o SQL Server",
         "O banco roda no container <code>pnad-sqlserver</code>. A senha vem do <code>.env</code>, o mesmo arquivo do docker-compose.")

**Por que pymssql?** Ele já traz o driver embutido. Com `pyodbc` seria preciso instalar o *ODBC Driver 18 for SQL Server* no Windows antes.

**Por que `autocommit`?** `CREATE DATABASE` não pode rodar dentro de uma transação, e as procedures do projeto já abrem e fecham as próprias transações. Deixar o driver abrir uma transação implícita por cima só atrapalharia.

In [3]:
try:
    db = Banco().conectar()
    versao = db.consultar("SELECT @@SERVERNAME AS servidor, SERVERPROPERTY('ProductVersion') AS versao, "
                          "SERVERPROPERTY('Edition') AS edicao, SYSDATETIME() AS agora")
    ui.tabela(versao, "Servidor conectado")
except Exception as erro:
    ui.aviso("Não consegui conectar",
             f"{mensagem_erro(erro)}<br>Confira se o container está de pé com <code>docker compose ps</code> "
             "e se o <code>.env</code> existe na raiz.", "erro")
    raise

servidor,versao,edicao,agora
33822ad8287a,b'16.0.4305.5',b'Developer Edition (64-bit)',2026-10-31 22:38:12.219727


In [4]:
ui.secao(2, "Arquitetura",
         "Seis etapas em sequência, todas gravando no mesmo log. Linhas inválidas não somem: vão para uma tabela própria com o motivo.")
ui.diagrama_arquitetura()

| Schema | Papel |
|---|---|
| `staging` | Dados como vieram do CSV, já com tipos, e as linhas rejeitadas |
| `dw` | Dimensões e fato (modelo estrela) |
| `etl` | Functions e procedures do processo |
| `auditoria` | Log de execução de todas as etapas |

O enunciado desenha o staging no Oracle. Como a disciplina é SQL Server, o staging virou um schema do mesmo banco, o que dispensa linked server entre os dois.

In [5]:
ui.secao(3, "Banco e schemas",
         "Arquivos <code>sql/00_setup</code>. Cria o banco com collation que guarda acento e compara sem diferenciar maiúscula e acento.")

- **`Latin1_General_CI_AI`**: o code page 1252 guarda os acentos do português em `VARCHAR`. *CI* (case insensitive) e *AI* (accent insensitive) fazem `'SAO PAULO'` ser igual a `'São Paulo'` nas comparações.
- **Recovery `SIMPLE`**: ambiente de estudo com cargas grandes. Evita o arquivo de log do banco crescer sem backup.
- **Schemas como pacotes**: o SQL Server não tem `PACKAGE` como o Oracle. Agrupar as rotinas no schema `etl` cumpre o mesmo papel de organização.

O separador `GO` dos scripts é do SSMS e do sqlcmd, não do SQL Server. A função `executar_script` quebra o arquivo nos `GO` e manda lote por lote, igual ao sqlcmd.

In [6]:
for script in ["sql/00_setup/01_criar_banco.sql", "sql/00_setup/02_criar_schemas.sql"]:
    lotes = db.executar_script(script)
    print(f"{script}: {lotes} lote(s)")

db.usar_projeto()
ui.tabela(db.consultar("""
    SELECT name AS schema_nome, schema_id
    FROM sys.schemas
    WHERE name IN ('staging', 'dw', 'etl', 'auditoria')
    ORDER BY name;"""), "Schemas criados")

sql/00_setup/01_criar_banco.sql: 3 lote(s)
sql/00_setup/02_criar_schemas.sql: 3 lote(s)


schema_nome,schema_id
auditoria,8
dw,6
etl,7
staging,5


In [7]:
ui.secao(4, "Log de execução",
         "Arquivos <code>sql/01_auditoria</code>. Uma linha por execução de procedure, aberta no início e fechada no fim.")

Cada procedure chama `auditoria.usp_log_iniciar` (que grava a linha com status `EM EXECUCAO`) e, no fim, `auditoria.usp_log_finalizar` com:

- data e hora de início e fim
- quantidade de linhas lidas e gravadas
- status: `SUCESSO`, `SUCESSO PARCIAL` ou `ERRO`
- observação (competências, rejeitadas, mensagem de erro)

**Detalhe importante:** o log é gravado fora da transação da carga. Se a carga falhar e der `ROLLBACK`, a linha com o erro continua no log. É ela que conta o que aconteceu.

In [8]:
for script in ["sql/01_auditoria/01_tabela_log_execucao.sql", "sql/01_auditoria/02_usp_log.sql"]:
    db.executar_script(script)

ui.tabela(db.consultar("""
    SELECT COLUMN_NAME AS coluna, DATA_TYPE AS tipo, CHARACTER_MAXIMUM_LENGTH AS tamanho, IS_NULLABLE AS aceita_nulo
    FROM INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_SCHEMA = 'auditoria' AND TABLE_NAME = 'log_execucao'
    ORDER BY ORDINAL_POSITION;"""), "auditoria.log_execucao")

coluna,tipo,tamanho,aceita_nulo
id_log,int,NULL,NO
no_processo,varchar,100,NO
ds_arquivo,nvarchar,400,YES
dt_inicio,datetime2,NULL,NO
dt_fim,datetime2,NULL,YES
qt_linhas_lidas,int,NULL,YES
qt_linhas_gravadas,int,NULL,YES
st_execucao,varchar,20,NO
ds_observacao,varchar,2.000,YES
no_usuario,nvarchar,128,NO


In [9]:
ui.secao(5, "Functions de normalização textual",
         "Arquivo <code>sql/02_functions/01_fn_normalizacao_texto.sql</code>. Tira espaços extras, remove acentos e aplica Title Case.")

| Function | O que faz | Como |
|---|---|---|
| `fn_remover_espacos_extras` | `'  São   Paulo '` vira `'São Paulo'` | troca tab e quebra de linha por espaço, `TRIM` e `REPLACE` de espaço duplo em laço |
| `fn_remover_acentos` | `'São Luís'` vira `'Sao Luis'` | `TRANSLATE` com duas listas de 48 caracteres, posição a posição |
| `fn_capitular` | `'RIO DE JANEIRO'` vira `'Rio de Janeiro'` | percorre letra a letra e mantém *de, da, do, das, dos, e* em minúsculo |
| `fn_normalizar_texto` | aplica as três | nessa ordem |

**Por que `COLLATE Latin1_General_100_BIN2` dentro delas?** O banco é *accent insensitive*: para ele `'á'` e `'a'` são iguais. Numa troca de caracteres isso faria o `TRANSLATE` casar o caractere errado. A collation binária compara byte a byte e deixa o resultado previsível.

In [10]:
db.executar_script("sql/02_functions/01_fn_normalizacao_texto.sql")

ui.tabela(db.consultar("""
    SELECT entrada,
           etl.fn_remover_espacos_extras(entrada) AS sem_espacos,
           etl.fn_remover_acentos(entrada)        AS sem_acentos,
           etl.fn_capitular(entrada)              AS capitular,
           etl.fn_normalizar_texto(entrada)       AS normalizado
    FROM (VALUES (N'   são   JOÃO   da  barra '),
                 (N'RIO DE JANEIRO'),
                 (N'florianópolis'),
                 (N'de onde vem')) AS t (entrada);"""), "Antes e depois (evidência E04)")

entrada,sem_espacos,sem_acentos,capitular,normalizado
são JOÃO da barra,são JOÃO da barra,sao JOAO da barra,São João da Barra,Sao Joao da Barra
RIO DE JANEIRO,RIO DE JANEIRO,RIO DE JANEIRO,Rio de Janeiro,Rio de Janeiro
florianópolis,florianópolis,florianopolis,Florianópolis,Florianopolis
de onde vem,de onde vem,de onde vem,De Onde Vem,De Onde Vem


In [11]:
ui.secao(6, "Functions de mapeamento",
         "Arquivos <code>02_fn_mapeamentos.sql</code> e <code>03_fn_regras_dw.sql</code>. Traduzem os códigos do dicionário do IBGE.")

Uma function por domínio do enunciado: UF, município, sexo, cor ou raça, escolaridade, faixa etária, faixa de rendimento, atendimento SUS, internação e sintomas, mais o nome do mês.

Duas regras valem para todas:

1. **Código desconhecido vira `'Não informado'`**, nunca `NULL` nem erro. Se o IBGE usar um código novo, a linha continua no DW.
2. **Vazio tem significado.** Em rendimento, atendimento e internação, campo vazio quer dizer que a pergunta não se aplicava à pessoa, e por isso vira `'Não aplicável'`.

Na PNAD Covid a coluna `CAPITAL` traz o código da UF quando a pessoa mora na capital e vem vazia fora dela. Por isso o mapa de municípios é "código da UF para nome da capital", e `fn_municipio_dw` manda quem mora no interior para `Demais Municipios` da UF.

In [12]:
for script in ["sql/02_functions/02_fn_mapeamentos.sql", "sql/02_functions/03_fn_regras_dw.sql"]:
    db.executar_script(script)

ui.tabela(db.consultar("""
    SELECT c.codigo,
           etl.fn_cor_raca(c.codigo)          AS cor_raca,
           etl.fn_escolaridade(c.codigo)      AS escolaridade,
           etl.fn_faixa_rendimento(c.codigo)  AS faixa_rendimento,
           etl.fn_internado(c.codigo)         AS internado,
           etl.fn_sintoma(c.codigo)           AS sintoma
    FROM (VALUES (1), (2), (3), (5), (8), (9), (NULL)) AS c (codigo);"""), "Códigos traduzidos")

ui.tabela(db.consultar("""
    SELECT u.codigo, etl.fn_sigla_uf(u.codigo) AS uf, etl.fn_nome_uf(u.codigo) AS nome_uf,
           etl.fn_nome_municipio(u.codigo) AS municipio_original,
           etl.fn_municipio_dw(u.codigo, u.codigo) AS municipio_no_dw,
           etl.fn_municipio_dw(u.codigo, NULL) AS fora_da_capital
    FROM (VALUES (21), (25), (31), (33), (53)) AS u (codigo);"""), "Território")

ui.tabela(db.consultar("""
    SELECT i.idade, etl.fn_faixa_etaria(i.idade) AS faixa_etaria
    FROM (VALUES (8), (17), (18), (29), (30), (59), (60), (91)) AS i (idade);"""), "Faixas etárias nos limites")

codigo,cor_raca,escolaridade,faixa_rendimento,internado,sintoma
1,Branca,Sem instrução,101 - 300,Sim,Sim
2,Preta,Fundamental incompleto,301 - 600,Não,Não
3,Amarela,Fundamental completo,601 - 800,Não foi atendido,Não sabe
5,Indígena,Médio completo,1.601 - 3.000,Não informado,Não informado
8,Não informado,"Pós-graduação, mestrado ou doutorado",50.001 - 100.000,Não informado,Não informado
9,Ignorado,Não informado,Mais de 100.000,Ignorado,Ignorado
NULL,Não informado,Não informado,Não aplicável,Não aplicável,Não aplicável


codigo,uf,nome_uf,municipio_original,municipio_no_dw,fora_da_capital
21,MA,Maranhão,Município de São Luís (MA),Sao Luis,Demais Municipios
25,PB,Paraíba,Município de João Pessoa (PB),Joao Pessoa,Demais Municipios
31,MG,Minas Gerais,Município de Belo Horizonte (MG),Belo Horizonte,Demais Municipios
33,RJ,Rio de Janeiro,Município de Rio de Janeiro (RJ),Rio de Janeiro,Demais Municipios
53,DF,Distrito Federal,Município de Brasília (DF),Brasilia,Demais Municipios


idade,faixa_etaria
8,Criança/Adolescente
17,Criança/Adolescente
18,Jovem Adulto
29,Jovem Adulto
30,Adulto
59,Adulto
60,Idoso
91,Idoso


In [13]:
ui.secao(7, "Estruturas da staging",
         "Arquivo <code>sql/03_staging/01_tabelas_staging.sql</code>. Cinco objetos, cada um com uma função na carga.")

| Objeto | Para que serve |
|---|---|
| `staging.csv_bruto` | cada linha do arquivo como texto, exatamente como veio |
| `staging.vw_csv_bruto` | view só com a coluna de texto: o `BULK INSERT` grava nela e o `IDENTITY` da tabela numera as linhas sozinho, sem precisar de format file |
| `staging.mapa_coluna_csv` | de-para entre a coluna do IBGE e a coluna da staging |
| `staging.pnad_covid_pessoa` | linhas válidas, com os tipos do enunciado |
| `staging.pnad_covid_rejeitado` | linhas que falharam, com o motivo e o texto original |

**Por que um de-para em tabela?** Os arquivos de meses diferentes não têm as mesmas colunas: o IBGE incluiu perguntas ao longo da pesquisa e as posições mudaram. Lendo pelo **nome** do cabeçalho, o mesmo código carrega qualquer mês. Se um nome mudar, basta atualizar a tabela.

In [14]:
db.executar_script("sql/03_staging/01_tabelas_staging.sql")
ui.tabela(db.consultar("SELECT nu_ordem, no_coluna_destino, no_coluna_origem, ds_pergunta FROM staging.mapa_coluna_csv ORDER BY nu_ordem;"),
          "staging.mapa_coluna_csv", max_linhas=30)

nu_ordem,no_coluna_destino,no_coluna_origem,ds_pergunta
1,NU_ANO,ANO,Ano de referência
2,NU_MES,V1013,Mês da pesquisa
3,UF,UF,Unidade da Federação
4,CO_MUNICIPIO,CAPITAL,Capital (código da UF quando mora na capital)
5,COR_RACA,A004,Cor ou raça
6,SEXO,A003,Sexo
7,IDADE,A002,Idade do morador
8,ESCOLARIDADE,A005,Escolaridade
9,FAIXA_RENDIMENTO,C01011,Faixa do rendimento/retirada em dinheiro
10,ATENDIMENTO_SUS,B0042,Local que buscou atendimento: pronto socorro do SUS/UPA


In [15]:
ui.secao(8, "Carga do CSV: etl.usp_carregar",
         "Arquivo <code>sql/03_staging/02_usp_carregar.sql</code>. Leitura, validação e gravação na staging, com log.")

A procedure trabalha em seis etapas:

1. **Leitura**: `BULK INSERT` do arquivo inteiro em `staging.csv_bruto`, uma linha do arquivo por registro. O caminho vem por parâmetro, então o comando é montado em SQL dinâmico.
2. **Cabeçalho**: acha a linha com `Ano`, `V1013` e `A002`, limpa aspas, `\r` do Windows e um possível BOM, e descobre a posição de cada coluna.
3. **Extração**: transforma cada linha em array JSON e usa `OPENJSON ... WITH` para tirar só as 23 colunas do de-para.
4. **Validação**: converte com `TRY_CAST` e junta todos os motivos de rejeição de cada linha com `CONCAT_WS`.
5. **Gravação**: numa transação, apaga a competência que está sendo recarregada, grava as válidas e as rejeitadas.
6. **Log**: lidas, gravadas, status e observação com competências e principal motivo de rejeição.

O truque do passo 3 fica mais claro com um exemplo. A célula abaixo faz com uma linha só o que a procedure faz com o arquivo inteiro:

In [16]:
ui.tabela(db.consultar("""
    DECLARE @linha VARCHAR(200) = '2020,31,31,,1,2,5';
    DECLARE @json  VARCHAR(400) = '["' + REPLACE(@linha, ',', '","') + '"]';

    SELECT @linha AS linha_csv, @json AS como_json, j.*
    FROM OPENJSON(@json)
         WITH (ano VARCHAR(10) '$[0]', uf VARCHAR(10) '$[1]',
               capital VARCHAR(10) '$[2]', mes VARCHAR(10) '$[6]') AS j;"""), "Uma linha do CSV vira colunas")

linha_csv,como_json,ano,uf,capital,mes
"2020,31,31,,1,2,5","[""2020"",""31"",""31"","""",""1"",""2"",""5""]",NULL,NULL,NULL,NULL
"2020,31,31,,1,2,5","[""2020"",""31"",""31"","""",""1"",""2"",""5""]",NULL,NULL,NULL,NULL
"2020,31,31,,1,2,5","[""2020"",""31"",""31"","""",""1"",""2"",""5""]",NULL,NULL,NULL,NULL
"2020,31,31,,1,2,5","[""2020"",""31"",""31"","""",""1"",""2"",""5""]",NULL,NULL,NULL,NULL
"2020,31,31,,1,2,5","[""2020"",""31"",""31"","""",""1"",""2"",""5""]",NULL,NULL,NULL,NULL
"2020,31,31,,1,2,5","[""2020"",""31"",""31"","""",""1"",""2"",""5""]",NULL,NULL,NULL,NULL
"2020,31,31,,1,2,5","[""2020"",""31"",""31"","""",""1"",""2"",""5""]",NULL,NULL,NULL,NULL


Repare em duas proteções que aparecem no código:

- `NULLIF(valor, '')` antes do `TRY_CAST`. Sem isso, `TRY_CAST('' AS INT)` devolve **0** e um campo vazio viraria um código válido.
- Linha que não vira JSON válido (barra invertida, aspas quebradas) recebe um array vazio no `OPENJSON` e é rejeitada como *mal formada*, em vez de derrubar a carga inteira.

Agora a carga de verdade, com o CSV de teste de `tests/dados`. Ele imita o arquivo do IBGE (cabeçalho entre aspas, colunas extras, quebras de linha do Windows) e traz **12 linhas válidas e 7 inválidas**, uma para cada regra.

In [17]:
db.executar_script("sql/03_staging/02_usp_carregar.sql")

# Recomeça do zero para o resultado bater com o esperado, mesmo rodando o notebook de novo.
limpar = db.consultar("SELECT OBJECT_ID(N'etl.usp_limpar_carga') AS id").iloc[0, 0]
if limpar is not None:
    db.executar("EXEC etl.usp_limpar_carga;")
else:
    db.executar("TRUNCATE TABLE staging.pnad_covid_pessoa; TRUNCATE TABLE staging.csv_bruto; DELETE FROM staging.pnad_covid_rejeitado;")

CSV_TESTE = "/var/opt/mssql/dados_teste/PNAD_COVID_TESTE.csv"   # caminho DENTRO do container
try:
    db.executar(f"EXEC etl.usp_carregar @caminho_arquivo = N'{CSV_TESTE}';")
    ui.aviso("Carga concluída", f"Arquivo <code>{CSV_TESTE}</code> lido pelo SQL Server.", "ok")
except Exception as erro:
    ui.aviso("A carga falhou", mensagem_erro(erro), "erro")

ui.tabela(db.log(1), "Log da carga (evidência E05)")

id_log,no_processo,dt_inicio,seg,lidas,gravadas,status,observacao
1,etl.usp_carregar,2026-10-31 22:40:07,0,847,0,ERRO,Competências: nenhuma | Rejeitadas: 847 | Principal motivo: linha mal formada (não foi possível separar os campos) (847) | Detalhes: staging.pnad_covid_rejeitado WHERE id_log = 1


In [18]:
ui.tabela(db.consultar("""
    SELECT NU_ANO, NU_MES, COUNT(*) AS linhas,
           SUM(CASE WHEN CO_MUNICIPIO IS NULL THEN 1 ELSE 0 END) AS fora_da_capital
    FROM staging.pnad_covid_pessoa
    GROUP BY NU_ANO, NU_MES ORDER BY NU_ANO, NU_MES;"""), "Staging por competência (evidência E06)")

ui.tabela(db.consultar("""
    SELECT nu_linha, ds_motivo, LEFT(ds_linha, 60) + '...' AS inicio_da_linha
    FROM staging.pnad_covid_rejeitado
    ORDER BY nu_linha;"""), "Linhas rejeitadas e motivo (evidência E07)")

nu_linha,ds_motivo,inicio_da_linha
2,linha mal formada (não foi possível separar os campos),"2020,21,21,,1,2,5,1,1110011,110015970,1,1,152539,38.43,70.12..."
2,linha mal formada (não foi possível separar os campos),"2020,21,21,,1,2,5,1,1110011,110015970,1,1,152539,38.43,70.12..."
2,linha mal formada (não foi possível separar os campos),"2020,21,21,,1,2,5,1,1110011,110015970,1,1,152539,38.43,70.12..."
2,linha mal formada (não foi possível separar os campos),"2020,21,21,,1,2,5,1,1110011,110015970,1,1,152539,38.43,70.12..."
2,linha mal formada (não foi possível separar os campos),"2020,21,21,,1,2,5,1,1110011,110015970,1,1,152539,38.43,70.12..."
2,linha mal formada (não foi possível separar os campos),"2020,21,21,,1,2,5,1,1110011,110015970,1,1,152539,38.43,70.12..."
2,linha mal formada (não foi possível separar os campos),"2020,21,21,,1,2,5,1,1110011,110015970,1,1,152539,38.43,70.12..."
2,linha mal formada (não foi possível separar os campos),"2020,21,21,,1,2,5,1,1110011,110015970,1,1,152539,38.43,70.12..."
2,linha mal formada (não foi possível separar os campos),"2020,21,21,,1,2,5,1,1110011,110015970,1,1,152539,38.43,70.12..."
2,linha mal formada (não foi possível separar os campos),"2020,21,21,,1,2,5,1,1110011,110015970,1,1,152539,38.43,70.12..."


In [19]:
ui.secao(9, "Modelo dimensional",
         "Arquivo <code>sql/04_dw/01_tabelas_dw.sql</code>. Estrela com três dimensões e uma fato.")

**Grão da fato:** uma linha por mês × município × perfil de pessoa.

A PNAD não traz identificador da pessoa. Por isso a `dim_pessoa` guarda **perfis**: cada combinação única de cor ou raça, sexo, faixa etária, escolaridade e faixa de rendimento. A fato soma quantas pessoas de cada perfil responderam e quantas tiveram sintoma, buscaram o SUS ou foram internadas.

Colunas acrescentadas ao enunciado:

| Coluna | Motivo |
|---|---|
| `dim_tempo.nu_mes` | ordenar os meses e achar a chave pelo número, não pelo nome |
| `fato_pnad_covid.qt_pessoas` | base para calcular percentuais |
| `staging.pnad_covid_pessoa.ID_LOG` | ligar cada linha à carga que a gravou |

A integridade fica no próprio banco: chaves primárias, `UNIQUE` nas chaves naturais das dimensões, FKs da fato para as dimensões e um `CHECK` que impede qualquer métrica maior que `qt_pessoas`.

In [20]:
db.executar_script("sql/04_dw/01_tabelas_dw.sql")

ui.tabela(db.consultar("""
    SELECT t.name AS tabela, c.name AS coluna, ty.name AS tipo,
           CASE WHEN ic.column_id IS NOT NULL THEN 'PK' ELSE '' END AS chave,
           ISNULL(OBJECT_NAME(fkc.referenced_object_id), '') AS referencia
    FROM sys.tables t
    JOIN sys.columns c ON c.object_id = t.object_id
    JOIN sys.types ty ON ty.user_type_id = c.user_type_id
    LEFT JOIN sys.indexes i ON i.object_id = t.object_id AND i.is_primary_key = 1
    LEFT JOIN sys.index_columns ic ON ic.object_id = t.object_id AND ic.index_id = i.index_id AND ic.column_id = c.column_id
    LEFT JOIN sys.foreign_key_columns fkc ON fkc.parent_object_id = t.object_id AND fkc.parent_column_id = c.column_id
    WHERE SCHEMA_NAME(t.schema_id) = 'dw'
    ORDER BY CASE WHEN t.name LIKE 'fato%' THEN 1 ELSE 0 END, t.name, c.column_id;"""), "Tabelas do schema dw", max_linhas=40)

tabela,coluna,tipo,chave,referencia
dim_municipio,sk_municipio,int,PK,
dim_municipio,sg_uf,char,,
dim_municipio,no_municipio,varchar,,
dim_pessoa,sk_pessoa,int,PK,
dim_pessoa,cor_raca,varchar,,
dim_pessoa,sexo,char,,
dim_pessoa,faixa_etaria,varchar,,
dim_pessoa,escolaridade,varchar,,
dim_pessoa,faixa_rendimento,varchar,,
dim_tempo,sk_tempo,int,PK,


In [21]:
ui.secao(10, "Carga das dimensões",
         "Arquivos <code>sql/05_procedures_dw/01 a 03</code>. Cada uma insere só o que ainda não existe.")

As três seguem o mesmo padrão, e ele é a parte de performance mais importante do projeto:

1. **Reduz primeiro**: `SELECT DISTINCT` dos códigos na staging. Milhões de linhas viram poucas combinações (umas 54 de município, alguns milhares de perfis).
2. **Aplica as functions depois**, só nessas combinações. Chamar uma function escalar 54 vezes em vez de milhões é a diferença entre milissegundos e minutos.
3. **Insere com `NOT EXISTS`**: só registros novos, então rodar de novo não duplica nada.

| Procedure | Regras |
|---|---|
| `usp_carregar_dim_tempo` | ano e mês únicos, `no_mes` por `fn_nome_mes` |
| `usp_carregar_dim_municipio` | sigla da UF e nome normalizado por `fn_municipio_dw` |
| `usp_carregar_dim_pessoa` | faixa etária, sexo em M/F e descrições de cor, escolaridade e rendimento |

In [22]:
for script in ["sql/05_procedures_dw/01_usp_carregar_dim_tempo.sql",
               "sql/05_procedures_dw/02_usp_carregar_dim_municipio.sql",
               "sql/05_procedures_dw/03_usp_carregar_dim_pessoa.sql",
               "sql/05_procedures_dw/04_usp_carregar_fato.sql"]:
    db.executar_script(script)

for procedure in ["etl.usp_carregar_dim_tempo", "etl.usp_carregar_dim_municipio", "etl.usp_carregar_dim_pessoa"]:
    db.executar(f"EXEC {procedure};")

ui.tabela(db.log(3), "Log das dimensões")
ui.tabela(db.consultar("SELECT * FROM dw.dim_tempo ORDER BY nu_ano, nu_mes;"), "dw.dim_tempo")
ui.tabela(db.consultar("SELECT * FROM dw.dim_municipio ORDER BY sg_uf, no_municipio;"), "dw.dim_municipio")
ui.tabela(db.consultar("SELECT * FROM dw.dim_pessoa ORDER BY sk_pessoa;"), "dw.dim_pessoa")

id_log,no_processo,dt_inicio,seg,lidas,gravadas,status,observacao
4,etl.usp_carregar_dim_pessoa,2026-10-31 22:40:58,0,0,0,SUCESSO,0 perfil(is) distintos na staging; 0 novo(s); 0 já existia(m).
3,etl.usp_carregar_dim_municipio,2026-10-31 22:40:58,0,0,0,SUCESSO,0 município(s) distintos na staging; 0 novo(s); 0 já existia(m).
2,etl.usp_carregar_dim_tempo,2026-10-31 22:40:58,0,0,0,SUCESSO,0 competência(s) na staging; 0 nova(s); 0 já existia(m).


In [23]:
ui.secao(11, "Carga da fato e integridade",
         "Arquivo <code>sql/05_procedures_dw/04_usp_carregar_fato.sql</code>. Resolve as chaves, agrega e garante que nada fique órfão.")

`usp_carregar_fato` monta dois mapas pequenos, (UF, capital) → `sk_municipio` e (códigos da pessoa) → `sk_pessoa`, e liga a staging a eles. Um detalhe: `NULL` não é igual a `NULL` num `JOIN`, então códigos vazios viram `-1` dos dois lados.

As métricas:

| Métrica | Regra |
|---|---|
| `qt_pessoas` | `COUNT(*)` |
| `atendimento_SUS` | `ATENDIMENTO_SUS = 1` (buscou pronto socorro do SUS/UPA) |
| `internado` | `INTERNADO = 1` |
| `sintoma` | `fn_teve_sintoma(...) = 1`: ao menos um dos 12 sintomas |

Linha sem chave em alguma dimensão **não entra** na fato. Ela é contada no log e o status vira `SUCESSO PARCIAL`. Recarregar um mês apaga a versão anterior dele antes de inserir.

In [24]:
db.executar("EXEC etl.usp_carregar_fato;")
ui.tabela(db.log(1), "Log da fato")

ui.tabela(db.consultar("""
    SELECT st.qt AS pessoas_staging, ft.qt AS pessoas_fato,
           st.sus AS sus_staging, ft.sus AS sus_fato,
           st.internado AS internado_staging, ft.internado AS internado_fato
    FROM (SELECT COUNT(*) AS qt,
                 SUM(CASE WHEN ATENDIMENTO_SUS = 1 THEN 1 ELSE 0 END) AS sus,
                 SUM(CASE WHEN INTERNADO = 1 THEN 1 ELSE 0 END) AS internado
          FROM staging.pnad_covid_pessoa) st
    CROSS JOIN (SELECT SUM(qt_pessoas) AS qt, SUM(atendimento_SUS) AS sus, SUM(internado) AS internado
                FROM dw.fato_pnad_covid) ft;"""), "Reconciliação staging x fato (evidência E09)",
          rodape="Cada par de colunas precisa ter o mesmo valor.")

ui.tabela(db.consultar("""
    SELECT
      (SELECT COUNT(*) FROM dw.fato_pnad_covid f WHERE NOT EXISTS (SELECT 1 FROM dw.dim_tempo d WHERE d.sk_tempo = f.sk_tempo)) AS orfaos_tempo,
      (SELECT COUNT(*) FROM dw.fato_pnad_covid f WHERE NOT EXISTS (SELECT 1 FROM dw.dim_municipio d WHERE d.sk_municipio = f.sk_municipio)) AS orfaos_municipio,
      (SELECT COUNT(*) FROM dw.fato_pnad_covid f WHERE NOT EXISTS (SELECT 1 FROM dw.dim_pessoa d WHERE d.sk_pessoa = f.sk_pessoa)) AS orfaos_pessoa;"""),
          "Integridade referencial (evidência E10)", rodape="Os três precisam ser zero.")

id_log,no_processo,dt_inicio,seg,lidas,gravadas,status,observacao
5,etl.usp_carregar_fato,2026-10-31 22:41:12,0,0,0,ERRO,Linhas da staging: 0 | Pessoas agregadas na fato: 0 | Linhas na fato (grão mês x município x perfil): 0 | Staging vazia.


pessoas_staging,pessoas_fato,sus_staging,sus_fato,internado_staging,internado_fato
0,NULL,NULL,NULL,NULL,NULL


orfaos_tempo,orfaos_municipio,orfaos_pessoa
0,0,0


In [25]:
ui.secao(12, "Orquestração e teste automatizado",
         "Arquivo <code>sql/06_orquestracao</code> e <code>tests/sql/validar_carga_teste.sql</code>.")

`etl.usp_executar_pipeline` roda as etapas na ordem e, se alguma cair, registra no log em qual parou. `etl.usp_limpar_carga` esvazia staging, dimensões e fato (o log fica, porque ele é a evidência).

O teste compara o resultado da carga do CSV de teste com **28 valores esperados**: contagens do log, da staging, de cada motivo de rejeição, das dimensões e das métricas da fato. Ele foi gerado por `tests/gerar_teste.py` a partir dos mesmos dados do CSV, então o esperado nunca fica desatualizado.

No `scripts/testar` o arquivo interrompe com erro se algo divergir. Aqui o notebook roda só a parte que monta a tabela de verificações e mostra o resultado de cada uma.

In [26]:
db.executar_script("sql/06_orquestracao/01_usp_executar_pipeline.sql")

script = (RAIZ / "tests/sql/validar_carga_teste.sql").read_text(encoding="utf-8")
verificacoes_sql = script.split("IF EXISTS (SELECT 1 FROM @resultado")[0]
resultado = db.executar(verificacoes_sql)[-1]

falhas = resultado[resultado["situacao"] != "OK"]
if falhas.empty:
    ui.aviso("Todas as verificações passaram", f"{len(resultado)} de {len(resultado)} conferem com o esperado.", "ok")
else:
    ui.aviso("Há verificações falhando", f"{len(falhas)} de {len(resultado)} divergiram. Veja a tabela.", "erro")
ui.tabela(resultado, "Teste de ponta a ponta (evidência E11)", max_linhas=40)

verificacao,esperado,obtido,situacao
log usp_carregar: linhas lidas,19,847,FALHOU
log usp_carregar: linhas gravadas,12,0,FALHOU
log usp_carregar: status SUCESSO PARCIAL (1 = sim),1,0,FALHOU
staging: linhas,12,0,FALHOU
staging: faixa de rendimento '04' lida como 4,3,0,FALHOU
rejeitadas: linhas,7,847,FALHOU
rejeitadas: UF inexistente,1,0,FALHOU
rejeitadas: IDADE,1,0,FALHOU
rejeitadas: NU_MES,1,0,FALHOU
rejeitadas: SEXO,1,0,FALHOU


In [27]:
ui.secao(13, "Carga dos dados reais do IBGE",
         "Opcional. Baixa os microdados, carrega cada mês e roda o pipeline completo.")

Mude `CARREGAR_DADOS_REAIS` para `True` e escolha os meses. Cada mês tem entre 9 e 12 MB compactado e uns 380 mil entrevistados. A carga de um mês leva alguns minutos, dependendo da máquina.

Os CSVs ficam em `dados/`, que o docker-compose monta dentro do container em `/var/opt/mssql/dados`. Por isso o caminho passado para a procedure é o do container, não o do seu disco.

Esta célula limpa o resultado do teste antes de carregar. Depois dela, as partes 14 e as consultas de evidência mostram os dados reais.

In [30]:
CARREGAR_DADOS_REAIS = True
MESES = ["05"]          # de "05" a "11" (maio a novembro de 2020)

import urllib.request
import zipfile

URL_BASE = ("https://ftp.ibge.gov.br/Trabalho_e_Rendimento/"
            "Pesquisa_Nacional_por_Amostra_de_Domicilios_PNAD_COVID19/Microdados/Dados")
PASTA_DADOS = RAIZ / "dados"
PASTA_NO_SERVIDOR = "/var/opt/mssql/dados"

if not CARREGAR_DADOS_REAIS:
    ui.aviso("Carga real desligada", "Mude <code>CARREGAR_DADOS_REAIS = True</code> para baixar e carregar.", "info")
else:
    db.executar("EXEC etl.usp_limpar_carga;")
    for mes in MESES:
        nome_zip = f"PNAD_COVID_{mes}2020.zip"
        destino_zip = PASTA_DADOS / nome_zip
        if not any(PASTA_DADOS.glob(f"PNAD_COVID_{mes}2020*.csv")):
            print(f"Baixando {nome_zip}...")
            urllib.request.urlretrieve(f"{URL_BASE}/{nome_zip}", destino_zip)
            with zipfile.ZipFile(destino_zip) as z:
                for interno in z.namelist():
                    if interno.lower().endswith(".csv"):
                        (PASTA_DADOS / Path(interno).name).write_bytes(z.read(interno))
            destino_zip.unlink()

        for csv in sorted(PASTA_DADOS.glob(f"PNAD_COVID_{mes}2020*.csv")):
            print(f"Carregando {csv.name} na staging...")
            try:
                db.executar(f"EXEC etl.usp_carregar @caminho_arquivo = N'{PASTA_NO_SERVIDOR}/{csv.name}';")
            except Exception as erro:
                ui.aviso(f"Falha em {csv.name}", mensagem_erro(erro), "erro")

    print("Carregando dimensões e fato...")
    db.executar("EXEC etl.usp_executar_pipeline;")
    ui.tabela(db.log(10), "Log da carga real")

Baixando PNAD_COVID_052020.zip...
Carregando PNAD_COVID_052020.csv na staging...


Carregando dimensões e fato...


OperationalError: (20047, b'DB-Lib error message 20047, severity 9:\nDBPROCESS is dead or not enabled\n')

In [ ]:
ui.secao(14, "O que o Data Warehouse responde",
         "Consultas sobre o modelo estrela. Com o CSV de teste os números são pequenos; com a carga real ganham sentido.")

As métricas contam **entrevistados**, sem o peso amostral da PNAD (`V1032`). Elas descrevem a amostra, não estimam a população do Brasil.

In [ ]:
import matplotlib.pyplot as plt
ui.estilo_graficos()

mensal = db.consultar("""
    SELECT t.nu_ano, t.nu_mes, t.no_mes,
           SUM(f.qt_pessoas) AS entrevistados,
           100.0 * SUM(f.sintoma)         / SUM(f.qt_pessoas) AS pct_sintoma,
           100.0 * SUM(f.atendimento_SUS) / SUM(f.qt_pessoas) AS pct_sus,
           100.0 * SUM(f.internado)       / SUM(f.qt_pessoas) AS pct_internado
    FROM dw.fato_pnad_covid f
    JOIN dw.dim_tempo t ON t.sk_tempo = f.sk_tempo
    GROUP BY t.nu_ano, t.nu_mes, t.no_mes
    ORDER BY t.nu_ano, t.nu_mes;""")

if mensal.empty:
    ui.aviso("Sem dados na fato", "Rode as partes 8 a 11 (ou a 13) antes.", "aviso")
else:
    for coluna in ["pct_sintoma", "pct_sus", "pct_internado"]:
        mensal[coluna] = mensal[coluna].astype(float)
    fig, ax = plt.subplots(figsize=(9, 3.6))
    for coluna, rotulo in [("pct_sintoma", "Teve algum sintoma"),
                           ("pct_sus", "Buscou pronto socorro do SUS"),
                           ("pct_internado", "Ficou internado")]:
        ax.plot(mensal["no_mes"], mensal[coluna], marker="o", linewidth=2, label=rotulo)
        ax.annotate(f"{mensal[coluna].iloc[-1]:.1f}%", (len(mensal) - 1, mensal[coluna].iloc[-1]),
                    textcoords="offset points", xytext=(8, -3), fontsize=9, color="#A1A1AA")
    ax.set_title("Evolução mensal, % dos entrevistados")
    ax.set_ylabel("%")
    ax.legend(loc="upper left", bbox_to_anchor=(0, -0.14), ncol=3, fontsize=9)
    plt.tight_layout()
    plt.show()
    ui.tabela(mensal.round(2), "Números do gráfico")

In [ ]:
capitais = db.consultar("""
    SELECT TOP (10) m.sg_uf + ' · ' + m.no_municipio AS capital,
           SUM(f.qt_pessoas) AS entrevistados,
           100.0 * SUM(f.sintoma) / SUM(f.qt_pessoas) AS pct_sintoma
    FROM dw.fato_pnad_covid f
    JOIN dw.dim_municipio m ON m.sk_municipio = f.sk_municipio
    WHERE m.no_municipio <> 'Demais Municipios'
    GROUP BY m.sg_uf, m.no_municipio
    ORDER BY pct_sintoma DESC;""")

if not capitais.empty:
    capitais["pct_sintoma"] = capitais["pct_sintoma"].astype(float)
    ordenado = capitais.sort_values("pct_sintoma")
    fig, ax = plt.subplots(figsize=(9, 0.42 * len(ordenado) + 1))
    barras = ax.barh(ordenado["capital"], ordenado["pct_sintoma"], color=ui.PALETA[0], height=0.6)
    ax.bar_label(barras, fmt="%.1f%%", padding=4, fontsize=9, color="#A1A1AA")
    ax.set_title("Capitais com maior % de entrevistados com sintoma")
    ax.grid(axis="y", visible=False)
    ax.set_xlabel("%")
    plt.tight_layout()
    plt.show()

In [ ]:
ui.tabela(db.consultar("""
    SELECT p.faixa_rendimento,
           SUM(f.qt_pessoas)      AS entrevistados,
           SUM(f.sintoma)         AS com_sintoma,
           SUM(f.atendimento_SUS) AS buscaram_sus
    FROM dw.fato_pnad_covid f
    JOIN dw.dim_pessoa p ON p.sk_pessoa = f.sk_pessoa
    GROUP BY p.faixa_rendimento
    ORDER BY p.faixa_rendimento;"""), "Busca pelo SUS por faixa de rendimento")

In [ ]:
ui.secao(15, "Evidências para a entrega",
         "Onde está cada print pedido no roteiro de <code>docs/evidencias/README.md</code>.")

| Código | Evidência | Onde |
|---|---|---|
| E01 | Container rodando | terminal: `docker compose ps` |
| E02 | Instalação | terminal: `scripts/instalar` (ou partes 3 a 12 deste notebook) |
| E03 | Objetos por schema | `sql/07_consultas/01_evidencias.sql` |
| E04 | Normalização textual | parte 5 |
| E05 | Log de execução | partes 8, 10, 11 e 13 |
| E06 | Staging por competência | parte 8 |
| E07 | Rejeitadas | parte 8 |
| E08 | Dimensões | parte 10 |
| E09 | Reconciliação da fato | parte 11 |
| E10 | Integridade referencial | parte 11 |
| E11 | Teste automatizado | parte 12 |

In [ ]:
ui.tabela(db.log(20), "Histórico completo desta sessão", max_linhas=20)
db.conexao.close()
ui.aviso("Fim", "Conexão encerrada. O banco continua no container para as consultas no SSMS ou Azure Data Studio.", "ok")